# Laundering Intent: local-style judge on a free Colab GPU

Runs an open-weight judge through Ollama on Colab's GPU, with the same scripts and pinning as on the laptop.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Then run the cells top to bottom.

You will upload 4 files in step 2: `score_any.py`, `extra_monitors.py`, `scored_monitor.py`, `yb_traces.csv`.
Do not commit `yb_traces.csv` anywhere public (no licence).

In [ ]:
# 1. Install Ollama and start it in the background
!sudo apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
server = subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
!curl -s localhost:11434/api/version

In [ ]:
# 2. Upload the four files
from google.colab import files
up = files.upload()
print(sorted(up))

In [ ]:
# 3. Download the judge (about 8 GB for gemma3:12b) and show its exact build
JUDGE = "gemma3:12b"
!ollama pull {JUDGE}
!ollama show {JUDGE}
!pip -q install numpy scikit-learn

In [ ]:
# 4. Score the Yong & Bach traces. Resumable: if Colab disconnects, rerun cells 1, 3 and 4
#    after re-uploading scores_extra.csv along with the other files.
!python3 score_any.py --file yb_traces.csv --dataset yb --label-col class \
    --judge ollama/{JUDGE} --monitors naive,selfperm_general,placebo_general --out scores_extra.csv

In [ ]:
# 5. Download the results. Send this file to Saumya / Claude for analysis.
files.download("scores_extra.csv")